# Domácí úkol ke cvičení 3: Malý výrobní projekt v PuLP

Zadání.

Tento domácí úkol procvičuje samostatné sestavení, ověření a porovnání výrobních plánů. Otevřete notebook v Jupyteru nebo Google Colabu a spouštějte buňky shora dolů pomocí Shift+Enter. Používáme Python 3.12 nebo novější a PuLP 3.3.2. Nejprve spusťte instalační buňku ze C2 v sekci 0.

V zadání doplňujte buňky označené **Doplňte** a komentáře `TODO`. Samotné spuštění nevyplněné šablony model nevyřeší. Před opětovným sestavením modelu spusťte znovu buňku, která jej vytváří; před odevzdáním restartujte jádro a spusťte celý doplněný notebook shora. Ve verzi s řešením jsou všechny kroky vyplněné. Data jsou umělá a úplně uvedená níže.

## 0. Příprava prostředí — spusťte jako první
Tato buňka nainstaluje **PuLP 3.3.2** do prostředí aktuálního notebooku, pokud
správná verze chybí. Stačí ji spustit; příkazy není potřeba upravovat.

Pokud už máte v paměti PuLP 4, buňka po instalaci výpočet zastaví a vyzve
k **restartu relace/jádra**. V Colabu otevřete nabídku **Runtime (Běhové prostředí)**
a zvolte restart relace; v Jupyteru **Kernel → Restart Kernel**. Potom spusťte
notebook znovu shora. Samotné opakování `import pulp` starou verzi z paměti neodstraní.
Po hlášení „Připraveno“ můžete pokračovat. Instalace vyžaduje připojení k internetu.

In [1]:
import sys
import subprocess
from importlib.metadata import version, PackageNotFoundError

REQUIRED_PULP = "3.3.2"
try:
    installed_pulp = version("pulp")
except PackageNotFoundError:
    installed_pulp = None

if installed_pulp != REQUIRED_PULP:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "--quiet",
        "--disable-pip-version-check", "--no-cache-dir", "pulp==3.3.2"
    ])
if version("pulp") != REQUIRED_PULP:
    raise RuntimeError("PuLP 3.3.2 se nenainstalovalo. Zkontrolujte výpis instalace výše.")

loaded_pulp = sys.modules.get("pulp")
if loaded_pulp is not None and getattr(loaded_pulp, "__version__", None) != REQUIRED_PULP:
    raise RuntimeError(
        "PuLP 3.3.2 je nainstalované, ale v paměti zůstává jiná verze. "
        "V Colabu zvolte Runtime > Restart session (případně Restart runtime). "
        "V Jupyteru zvolte Kernel > Restart Kernel. Potom spusťte všechny buňky shora."
    )
print("Připraveno: PuLP 3.3.2 je nainstalované. Pokračujte dalšími buňkami.")

Připraveno: PuLP 3.3.2 je nainstalované. Pokračujte dalšími buňkami.


In [2]:
import pulp

if pulp.__version__ != "3.3.2":
    raise RuntimeError("Spusťte sekci 0 a postupujte podle pokynu k restartu.")
print("PuLP:", pulp.__version__)
TOL = 1e-7

PuLP: 3.3.2


## 1. Zadání: co máme rozhodnout?
Malá výrobna připravuje tři kapalné směsi **A, B a C**. Rozhodujeme, kolik litrů každé směsi vyrobit v jedné dávce. K dispozici je dohromady **18 minut stroje** a **16 kg společné suroviny**.

| Údaj | A | B | C |
|---|---:|---:|---:|
| Příspěvek k zisku [Kč/l] | 40 | 30 | 50 |
| Čas stroje [min/l] | 2 | 1 | 3 |
| Spotřeba suroviny [kg/l] | 1 | 2 | 2 |
| Nejvyšší odbyt [l] | 4 | 6 | 5 |

Směsi jsou dělitelné: lze vyrobit i část litru. Vše vyrobené se prodá, nejvýše však do uvedené meze odbytu. Nevyrábíme do zásoby. Spotřeby i příspěvky jsou přímo úměrné množství. Příspěvek k zisku již zohledňuje náklady na daný litr; další náklady neuvažujeme. Data jsou **uměle vytvořená pro toto cvičení**.

**Doplňte:** Definujte proměnné včetně jednotek a oborů. Na papír nebo do následující buňky zapište účelovou funkci a všechna omezení. Proč nejvyšší odbyt není povinné výrobní množství?

**Vaše formulace:**

*Doplňte proměnné, obory, účelovou funkci a omezení.*

## 2. Data oddělená od modelu
Seznam `products` obsahuje názvy výrobků. Slovník přiřazuje klíči hodnotu: například `profit["A"]` je příspěvek k zisku z litru směsi A. Ve vnořeném slovníku `use` nejprve vybereme zdroj, potom výrobek: `use["machine"]["A"]` je počet minut stroje na litr A.

**Doplňte ústně:** V jakých jednotkách jsou `capacity["material"]`, `use["material"]["B"]` a `demand["B"]`? Která čísla jsou společná pro celou dávku?

In [3]:
products = ["A", "B", "C"]
resources = ["machine", "material"]

profit = {"A": 40, "B": 30, "C": 50}  # Kč/l
use = {
    "machine": {"A": 2, "B": 1, "C": 3},  # min/l
    "material": {"A": 1, "B": 2, "C": 2},  # kg/l
}
capacity = {"machine": 18, "material": 16}
units = {"machine": "min", "material": "kg"}
demand = {"A": 4, "B": 6, "C": 5}  # l

print(profit["A"])
print(use["machine"]["A"])

40
2


## 3. Proměnné a účelová funkce
Vytvoříme nový model a prázdný slovník `x`. Cyklus pro každý výrobek vytvoří jednu proměnnou. K `x["A"]` pak přistupujeme stejně jako k údaji `profit["A"]`; v `x` jsou ale objekty proměnných, jejichž hodnoty teprve hledáme.

In [4]:
model = pulp.LpProblem("production", pulp.LpMaximize)
x = {}
for p in products:
    x[p] = pulp.LpVariable(f"x_{p}", lowBound=0, cat="Continuous")

print(x)

{'A': x_A, 'B': x_B, 'C': x_C}


Výraz `profit[p] * x[p]` je příspěvek výrobku `p`. Zápis
`pulp.lpSum(profit[p] * x[p] for p in products)` tyto členy sečte přes celý seznam výrobků.

**Doplňte:** Přidejte do modelu účelovou funkci. Výraz má odpovídat součtu $40x_A+30x_B+50x_C$. Použijte `lpSum`, aby zápis fungoval i při změně seznamu výrobků.

In [5]:
# TODO: model += ... , "profit"

## 4. Od jednoho omezení k celé skupině
Nejprve sestavíme pouze výraz pro spotřebu stroje. Následující buňka jej **nepřidává** jako omezení; jen ukazuje tři členy součtu.

In [6]:
machine_use = 2*x["A"] + x["B"] + 3*x["C"]
print(machine_use)

2*x_A + x_B + 3*x_C


**Doplňte:** Tentýž výraz sestavte pomocí `use["machine"][p]` a `lpSum` a vypište jej. Potom nahraďte klíč `"machine"` proměnnou `r`: cyklem přes `resources` přidejte pro každý zdroj jednu kapacitní podmínku. Pravá strana je `capacity[r]`. Omezení pojmenujte `f"capacity_{r}"`.

Druhým cyklem přidejte pro každý výrobek podmínku `x[p] <= demand[p]`, pojmenovanou `f"sales_{p}"`. Nezápornost už zajišťuje `lowBound=0`. V modelu tak bude pět pojmenovaných omezení a tři nezáporné proměnné.

In [7]:
# TODO: Vypište spotřebu stroje sestavenou pomocí lpSum.
# TODO: Cyklus přes resources: přidejte obě kapacitní podmínky.
# TODO: Cyklus přes products: přidejte tři meze odbytu.

## 5. Řešení a první čtení výsledku
**Doplňte:** Vypište model a porovnejte jej s formulací z první sekce. Potom jej vyřešte pomocí `pulp.PULP_CBC_CMD(msg=False)` a vypište `pulp.LpStatus[model.status]`.

Pro tuto malou úlohu pokračujte k optimálnímu plánu pouze při `model.status == pulp.LpStatusOptimal`. Jinak výpočet zastavte pomocí `raise RuntimeError(...)` a vraťte se k modelu. Po úspěšném řešení uložte číselné hodnoty do slovníku `plan` a hodnotu účelové funkce do `base_profit`. Symbolické proměnné `x` a nalezená čísla `plan` jsou dvě různé věci.

In [8]:
# TODO: Vypište model, vyřešte jej a ověřte stav.
# TODO: plan = {p: x[p].value() for p in products}
# TODO: Uložte a vypište base_profit a plán v litrech.

## 6. Kontrola původních podmínek
Stav `Optimal` je informace od řešiče o zadaném modelu. Ještě ověříme, zda nalezená čísla splňují **původní podmínky zadání**. Kontrolu počítáme ze vstupních dat a hodnot `plan`, nikoli jen z interních údajů o omezeních v PuLP. Přitom musíme porovnat i slovníky s tabulkou zadání; chybný údaj převzatý do modelu i kontroly by samotný přepočet neodhalil.

**Doplňte:** Pro každý zdroj sestavte řádek tabulky: spotřeba, kapacita, rezerva, jednotka a splnění podmínky. Rezerva je kapacita minus spotřeba. Následně ověřte meze `0 <= plan[p] <= demand[p]` a znovu spočítejte příspěvek k zisku.

Zde už sčítáme čísla, proto stačí vestavěné `sum(...)`. Použijte absolutní toleranci `TOL = 1e-7`: pro kapacity v minutách, respektive kg, pro množství v litrech a pro shodu hodnot účelové funkce v Kč. Pro jiná měřítka dat je nutné toleranci zvolit znovu. Nezaokrouhlujte hodnoty před kontrolou; dvě desetinná místa použijte jen při výpisu. `assert podmínka` zastaví buňku, pokud podmínka není splněna.

In [9]:
# TODO: Pro každý r spočítejte sum(use[r][p] * plan[p] for p in products).
# Vypište spotřebu, kapacitu, rezervu, jednotku a výsledek kontroly s tolerancí.
# TODO: Ověřte nezápornost a mez odbytu u všech tří výrobků.
# TODO: Přepočtěte příspěvek k zisku a porovnejte jej s base_profit.

**Interpretace:** Které zdroje jsou plně využité? U kterých výrobků zbývá prostor do meze odbytu? Zaručuje samotné splnění kontrol optimalitu?

*Doplňte svůj komentář ke kontrolní tabulce.*

## 7. Samostatná změna: více času stroje
Výrobna může pro stejnou dávku využít **22 minut stroje místo 18**. Ostatní údaje zůstávají stejné. Zatím neuvažujeme cenu za dodatečný čas.

**Nejprve odhadněte:** Může se nejlepší dosažitelný příspěvek k zisku zhoršit? Musí se zvýšit výroba všech směsí? Zdůvodněte svou odpověď ještě před výpočtem.

**Doplňte:** Vytvořte `scenario_capacity = capacity.copy()` a změňte pouze položku `"machine"`. Kopie zachová data původní varianty. Sestavte **nový model** `scenario_model` s novými proměnnými `scenario_x` stejným postupem jako výše, vyřešte jej a ověřte stav. Uložte `scenario_plan` a `scenario_profit`. Původní `model`, `plan` a `capacity` zachovejte pro porovnání.

**Můj odhad:** *Doplňte před spuštěním nové varianty.*

In [10]:
# TODO: Zkopírujte kapacity a změňte stroj na 22 minut.
# TODO: Nový model, nové proměnné, stejná účelová funkce, kapacity podle scenario_capacity a meze odbytu.
# TODO: Vyřešte, ověřte stav, uložte scenario_plan a scenario_profit.

**Doplňte kontrolu nové varianty:** Zopakujte tabulku ze sekce 6 s `scenario_plan`, `scenario_capacity` a `scenario_model`. Použijete-li omylem původní kapacitu, budete kontrolovat jinou úlohu. Potom porovnejte obě varianty v jedné tabulce.

In [11]:
# TODO: Zopakujte všechny kontroly s daty a řešením nové varianty.
# TODO: Vypište původní a nové množství každého výrobku, jejich rozdíl a oba příspěvky k zisku.

**Komentář:** Jak se změnila skladba výroby? Proč neporovnáváme jen hodnotu účelové funkce? Co ještě potřebujeme vědět, abychom rozhodli o pořízení dodatečného času stroje?

*Doplňte závěr z porovnání obou variant.*

## 8. Krátký závěr projektu
**Doplňte vlastními slovy** krátký závěr, který obsahuje doporučený plán, jeho kontrolu a srovnání s novou variantou. Připojte předpoklady, které omezují použití výsledku, zdroj dat a návod k opakování výpočtu. Neopisujte jen výstup `Optimal`.

Při přípravě vlastního zápočtového projektu lze použít stejnou stavbu: **zadání a model → výpočet → kontrola a interpretace → jedna změna zadání**. Samostatný projekt musí obsahovat vaše vlastní zpracování a uvedení použitých zdrojů a pomoci.

*Zde doplňte závěr, zdroje a návod ke spuštění.*

In [12]:
# Verze prostředí a řešiče pro záznam výpočtu.
print("Python:", sys.version.split()[0])
print("PuLP:", pulp.__version__)
cbc_info = subprocess.run(
    [pulp.PULP_CBC_CMD().path, "-stop"],
    capture_output=True, text=True, check=True,
)
for line in cbc_info.stdout.splitlines():
    if "Version:" in line:
        print("CBC", line.strip())

Python: 3.12.14
PuLP: 3.3.2
CBC Version: 2.10.3
